# Fine-Tuning do Mistral 7B para Assistente Médico (Local)

**Tech Challenge Fase 3 - FIAP Pós-Graduação em IA para Desenvolvedores**

Este notebook realiza o fine-tuning do modelo Mistral 7B Instruct v0.3 usando:
- **QLoRA 4-bit**: Quantização para caber em GPUs com 16GB+ VRAM
- **Unsloth**: Biblioteca otimizada para fine-tuning 2x mais rápido
- **Trainer HuggingFace**: Treinamento estável e compatível
- **Dataset**: MedQuAD híbrido (~20k pares QA: 80% PT + 20% EN + BR)

## Configuração para Execução Local

- **GPU**: NVIDIA com 16GB+ VRAM (RTX 3090, 4090, A100, etc.)
- **RAM**: 32GB+ recomendado
- **Disco**: ~20GB para modelo e checkpoints

## Fluxo do Notebook

1. Verificar ambiente e GPU
2. Configurar paths locais
3. Carregar dataset
4. Carregar modelo base (Mistral 7B)
5. Configurar LoRA
6. Treinar (com checkpoints)
7. Salvar modelo
8. Testar inferência

---
## 1. Verificar Ambiente

Verificamos se as dependências estão instaladas e se há GPU disponível.

**Pré-requisitos (instalar antes de rodar):**
```bash
pip install unsloth peft accelerate bitsandbytes datasets transformers torch
```

In [1]:
# ============================================================================
# VERIFICAR DEPENDÊNCIAS
# ============================================================================
# Usa importlib.util para verificar sem importar (evita conflitos de ordem)
# ============================================================================

import importlib.util

def check_package_installed(package_name):
    """Verifica se pacote está instalado sem importar."""
    spec = importlib.util.find_spec(package_name)
    return spec is not None

packages = ['unsloth', 'torch', 'torchvision', 'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes']
missing = [p for p in packages if not check_package_installed(p)]

if missing:
    print(f"✗ Pacotes faltando: {', '.join(missing)}")
    print("")
    print("Instale com:")
    print(f"  pip install {' '.join(missing)}")
else:
    print("✓ Todas as dependências instaladas!")

✓ Todas as dependências instaladas!


In [2]:
# ============================================================================
# IMPORTAR UNSLOTH PRIMEIRO (OBRIGATÓRIO)
# ============================================================================
# Unsloth deve ser importado antes de transformers, peft, trl para aplicar
# todas as otimizações corretamente.
# ============================================================================

import unsloth  # DEVE ser a primeira importação!

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU disponível: {gpu_name}")
    print(f"  Memória total: {gpu_memory:.1f} GB")
    
    if gpu_memory < 15:
        print("")
        print("⚠️  AVISO: GPU com menos de 16GB de VRAM.")
        print("  Pode ser necessário reduzir batch_size ou max_seq_length.")
elif torch.backends.mps.is_available():
    print("✓ Apple Silicon (MPS) disponível")
    print("  Nota: Fine-tuning em MPS pode ser mais lento que CUDA")
else:
    print("✗ ERRO: Nenhuma GPU detectada!")
    print("  Fine-tuning requer GPU com 16GB+ de VRAM.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/nicola/.pyenv/versions/venv_fine_tuning/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ GPU disponível: NVIDIA RTX A5000
  Memória total: 25.3 GB


---
## 2. Configurar Paths Locais

Configuramos os caminhos relativos ao diretório do projeto.

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE PATHS LOCAIS
# ============================================================================

import os
from pathlib import Path

# Detectar diretório do projeto (assume que notebook está em notebooks/)
NOTEBOOK_DIR = Path(os.getcwd())
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR / "virtual-medical-assistant"

# Paths dos dados
DATA_DIR = PROJECT_DIR / "data" / "processed"
TRAINING_DATA_PATH = DATA_DIR / "training_data.jsonl"

# Paths dos modelos e checkpoints
MODELS_DIR = PROJECT_DIR / "models"
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"
FINAL_MODEL_DIR = MODELS_DIR / "medical-assistant-final"

# Criar diretórios se não existirem
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Configuração de paths:")
print(f"  Projeto: {PROJECT_DIR}")
print(f"  Dataset: {TRAINING_DATA_PATH}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Modelo final: {FINAL_MODEL_DIR}")

Configuração de paths:
  Projeto: /home/nicola
  Dataset: /home/nicola/data/processed/training_data.jsonl
  Checkpoints: /home/nicola/models/checkpoints
  Modelo final: /home/nicola/models/medical-assistant-final


In [4]:
# ============================================================================
# VERIFICAR SE O DATASET EXISTE
# ============================================================================

if TRAINING_DATA_PATH.exists():
    file_size = TRAINING_DATA_PATH.stat().st_size / (1024 * 1024)
    print(f"✓ Dataset encontrado: {TRAINING_DATA_PATH}")
    print(f"  Tamanho: {file_size:.2f} MB")
else:
    print(f"✗ ERRO: Dataset não encontrado!")
    print(f"  Esperado em: {TRAINING_DATA_PATH}")
    print("")
    print("  Para resolver, execute:")
    print("    python scripts/03_preparar_dataset.py")

✗ ERRO: Dataset não encontrado!
  Esperado em: /home/nicola/data/processed/training_data.jsonl

  Para resolver, execute:
    python scripts/03_preparar_dataset.py


---
## 3. Carregar e Preparar Dataset

O dataset está no formato Alpaca JSONL:
```json
{"instruction": "...", "input": "pergunta", "output": "resposta"}
```

In [5]:
# ============================================================================
# CARREGAR DATASET
# ============================================================================

from datasets import load_dataset

# Carregar do arquivo JSONL
dataset = load_dataset('json', data_files=str(TRAINING_DATA_PATH), split='train')

print(f"✓ Dataset carregado: {len(dataset)} registros")
print(f"  Colunas: {dataset.column_names}")
print("")
print("Exemplo de registro:")
print(f"  instruction: {dataset[0]['instruction'][:80]}...")
print(f"  input: {dataset[0]['input'][:80]}...")
print(f"  output: {dataset[0]['output'][:80]}...")

FileNotFoundError: Unable to find '/home/nicola/data/processed/training_data.jsonl'

In [ ]:
# ============================================================================
# DIVIDIR EM TREINO E VALIDAÇÃO
# ============================================================================

dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
eval_dataset = dataset_split['test']

print(f"✓ Dataset dividido:")
print(f"  Treino: {len(train_dataset)} registros (90%)")
print(f"  Validação: {len(eval_dataset)} registros (10%)")

In [ ]:
# ============================================================================
# ESTATÍSTICAS DO DATASET
# ============================================================================

import numpy as np

input_lengths = [len(x['input']) for x in dataset]
output_lengths = [len(x['output']) for x in dataset]
total_lengths = [len(x['input']) + len(x['output']) for x in dataset]

print("Estatísticas de comprimento (caracteres):")
print("")
print("  Perguntas (input):")
print(f"    Média: {np.mean(input_lengths):.0f}")
print(f"    Máximo: {np.max(input_lengths)}")
print(f"    Percentil 95: {np.percentile(input_lengths, 95):.0f}")
print("")
print("  Respostas (output):")
print(f"    Média: {np.mean(output_lengths):.0f}")
print(f"    Máximo: {np.max(output_lengths)}")
print(f"    Percentil 95: {np.percentile(output_lengths, 95):.0f}")

---
## 4. Carregar Modelo Base (Mistral 7B)

Usamos o **Unsloth** para carregar o modelo com:
- Quantização 4-bit (reduz de ~14GB para ~4GB de VRAM)
- Otimizações de velocidade (2x mais rápido)

In [ ]:
# ============================================================================
# CONFIGURAÇÕES DO MODELO
# ============================================================================

# Modelo base
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

# Tamanho máximo de sequência (em tokens)
MAX_SEQ_LENGTH = 2048

# Configuração de precisão
DTYPE = None  # Auto-detectar
LOAD_IN_4BIT = True  # Quantização 4-bit

print("Configuração do modelo:")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Quantização: 4-bit")

In [ ]:
# ============================================================================
# CARREGAR MODELO E TOKENIZER
# ============================================================================

from unsloth import FastLanguageModel

print("Carregando modelo (pode demorar alguns minutos)...")
print("")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("")
print("✓ Modelo carregado com sucesso!")

---
## 5. Configurar LoRA

**LoRA (Low-Rank Adaptation)** permite treinar apenas uma pequena fração dos parâmetros.

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DO LORA
# ============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # Rank do LoRA
    lora_alpha=128,          # Scaling factor
    lora_dropout=0.05,       # Dropout para regularização
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
percent = 100 * trainable_params / total_params

print(f"\n✓ LoRA configurado!")
print(f"  Parâmetros treináveis: {trainable_params:,} ({percent:.2f}%)")
print(f"  Parâmetros totais: {total_params:,}")

---
## 6. Preparar Formato de Treinamento

In [ ]:
# ============================================================================
# TEMPLATE ALPACA EM PORTUGUÊS
# ============================================================================

ALPACA_PROMPT = """Abaixo está uma instrução que descreve uma tarefa, junto com uma entrada que fornece contexto adicional. Escreva uma resposta que complete adequadamente a solicitação.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token

print("Template Alpaca configurado.")

In [ ]:
# ============================================================================
# FUNÇÃO DE FORMATAÇÃO DO DATASET
# ============================================================================

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = ALPACA_PROMPT.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

train_dataset_formatted = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatando treino"
)

eval_dataset_formatted = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatando validação"
)

print(f"\n✓ Dataset formatado!")
print(f"  Treino: {len(train_dataset_formatted)} exemplos")
print(f"  Validação: {len(eval_dataset_formatted)} exemplos")

---
## 7. Configurar Treinamento

Configuração para execução local (sem limite de sessão).

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DO TREINAMENTO (LOCAL)
# ============================================================================

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Detectar capacidade da GPU para ajustar batch size
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_memory >= 24:  # A100, RTX 4090, etc.
        BATCH_SIZE = 4
        GRAD_ACCUM = 4
    elif gpu_memory >= 16:  # RTX 3090, A10, etc.
        BATCH_SIZE = 2
        GRAD_ACCUM = 8
    else:  # GPUs menores
        BATCH_SIZE = 1
        GRAD_ACCUM = 16
else:
    BATCH_SIZE = 1
    GRAD_ACCUM = 16

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    # ===== Epochs e Batching =====
    num_train_epochs=5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # ===== Learning Rate =====
    learning_rate=2e-4,
    warmup_steps=200,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    # ===== Precisão =====
    fp16=torch.cuda.is_available(),

    # ===== Checkpoints =====
    save_strategy="steps",
    save_steps=1000,           # Menos frequente que Colab (sessão não expira)
    save_total_limit=3,

    # ===== Avaliação =====
    eval_strategy="steps",
    eval_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # ===== Logging =====
    logging_steps=50,
    report_to="none",

    # ===== Otimizações =====
    optim="adamw_8bit",
    seed=42,
)

print("✓ TrainingArguments configurado!")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Checkpoints: a cada {training_args.save_steps} steps")

In [ ]:
# ============================================================================
# CRIAR TRAINER
# ============================================================================

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizando datasets...")
train_tokenized = train_dataset_formatted.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text", "instruction", "input", "output"],
    desc="Tokenizando treino"
)
eval_tokenized = eval_dataset_formatted.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text", "instruction", "input", "output"],
    desc="Tokenizando validação"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
)

print("✓ Trainer criado!")

---
## 8. Verificar Checkpoints Existentes

In [ ]:
# ============================================================================
# ENCONTRAR ÚLTIMO CHECKPOINT
# ============================================================================

def find_latest_checkpoint(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        return None

    checkpoints = [
        d for d in checkpoint_dir.iterdir()
        if d.is_dir() and d.name.startswith("checkpoint-")
    ]

    if not checkpoints:
        return None

    checkpoints_sorted = sorted(
        checkpoints,
        key=lambda x: int(x.name.split("-")[1])
    )

    return str(checkpoints_sorted[-1])

latest_checkpoint = find_latest_checkpoint(CHECKPOINT_DIR)

if latest_checkpoint:
    print(f"🔄 Checkpoint encontrado!")
    print(f"   {latest_checkpoint}")
    print("")
    print("   O treino será RETOMADO deste ponto.")
else:
    print("🆕 Nenhum checkpoint encontrado.")
    print("   O treino começará do zero.")

---
## 9. Treinar!

**Tempo estimado (GPU local):**
- RTX 4090 (24GB): ~3-4 horas para 5 epochs
- RTX 3090 (24GB): ~4-5 horas
- A100 (40GB): ~2-3 horas

In [ ]:
# ============================================================================
# TREINAR O MODELO
# ============================================================================

print("=" * 60)
print("  INICIANDO TREINAMENTO")
print("=" * 60)
print("")

if latest_checkpoint:
    print(f"🔄 Retomando de: {latest_checkpoint}")
    print("")
    trainer_stats = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("🆕 Iniciando treino do zero...")
    print("")
    trainer_stats = trainer.train()

print("")
print("=" * 60)
print("  TREINAMENTO CONCLUÍDO!")
print("=" * 60)

In [ ]:
# ============================================================================
# ESTATÍSTICAS DO TREINAMENTO
# ============================================================================

print("Estatísticas do treinamento:")
print(f"  Tempo total: {trainer_stats.metrics['train_runtime'] / 3600:.2f} horas")
print(f"  Steps totais: {trainer_stats.metrics['train_steps']}")
print(f"  Loss final: {trainer_stats.metrics['train_loss']:.4f}")
print(f"  Samples/segundo: {trainer_stats.metrics['train_samples_per_second']:.2f}")

---
## 10. Salvar Modelo Final

In [ ]:
# ============================================================================
# SALVAR MODELO FINAL
# ============================================================================

print(f"Salvando modelo em: {FINAL_MODEL_DIR}")
print("")

model.save_pretrained(str(FINAL_MODEL_DIR))
print("✓ Adapters LoRA salvos")

tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
print("✓ Tokenizer salvo")

print("")
print("Arquivos salvos:")
total_size = 0
for f in FINAL_MODEL_DIR.iterdir():
    size = f.stat().st_size / (1024 * 1024)
    total_size += size
    print(f"  {f.name}: {size:.2f} MB")

print(f"")
print(f"Tamanho total: {total_size:.2f} MB")

---
## 11. Testar Inferência

In [ ]:
# ============================================================================
# PREPARAR MODELO PARA INFERÊNCIA
# ============================================================================

FastLanguageModel.for_inference(model)
print("✓ Modelo preparado para inferência")

In [ ]:
# ============================================================================
# FUNÇÃO DE INFERÊNCIA
# ============================================================================

def ask_medical_question(question: str, max_new_tokens: int = 512) -> str:
    prompt = ALPACA_PROMPT.format(
        "Responda a seguinte pergunta médica de forma clara e detalhada.",
        question,
        ""
    )

    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
    )

    response = tokenizer.batch_decode(outputs)[0]

    if "### Resposta:" in response:
        response = response.split("### Resposta:")[1].strip()
        if EOS_TOKEN in response:
            response = response.split(EOS_TOKEN)[0].strip()

    return response

In [ ]:
# ============================================================================
# TESTAR COM PERGUNTAS MÉDICAS
# ============================================================================

perguntas_teste = [
    "Quais são os sintomas do diabetes tipo 2?",
    "Como é feito o diagnóstico de hipertensão arterial?",
    "Quais são os tratamentos disponíveis para asma?",
]

print("=" * 60)
print("  TESTE DE INFERÊNCIA")
print("=" * 60)

for i, pergunta in enumerate(perguntas_teste, 1):
    print(f"\n--- Pergunta {i} ---")
    print(f"Q: {pergunta}")
    print("")

    resposta = ask_medical_question(pergunta)

    print(f"R: {resposta}")
    print("-" * 40)

---
## 12. Carregar Modelo Salvo (Para Uso Futuro)

In [ ]:
# ============================================================================
# COMO CARREGAR O MODELO EM UMA SESSÃO FUTURA
# ============================================================================

# Descomente para usar:

# from unsloth import FastLanguageModel
# from pathlib import Path
#
# MODEL_PATH = Path("models/medical-assistant-final")
#
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=str(MODEL_PATH),
#     max_seq_length=2048,
#     dtype=None,
#     load_in_4bit=True,
# )
#
# FastLanguageModel.for_inference(model)
# print("✓ Modelo carregado e pronto para uso!")

print("Código para carregar modelo em sessão futura (descomente para usar)")

---
## Próximos Passos

Após o fine-tuning:

1. **Avaliação Baseline vs Fine-Tuned**
   - Execute `scripts/05_avaliar_baseline.py`
   - Execute `scripts/06_avaliar_finetuned.py`

2. **Integração com LangGraph**
   - O modelo será usado pelo agente `raciocinio` no grafo multi-agente

3. **Documentação**
   - Adicionar curvas de loss ao relatório LaTeX

In [ ]:
# ============================================================================
# FIM DO NOTEBOOK
# ============================================================================

print("🎉 Fine-tuning concluído com sucesso!")
print("")
print(f"Modelo salvo em: {FINAL_MODEL_DIR}")
print("")
print("Próximos passos:")
print("  1. Executar scripts de avaliação (05 e 06)")
print("  2. Integrar com LangGraph")
print("  3. Documentar resultados no relatório")